In [21]:
!pip install langchain langchain-community chromadb openai
!pip install -q langchain_huggingface sentence_transformers --no-deps
!pip install -U langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.2 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.3.4
    Uninstalling huggingface_hub-1.3.4:
      Successfully uninstalled huggingface_hub-1.3.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import glob

from langchain_community.document_loaders import PyPDFLoader

# Initialisation du loader de document pour charger un fichier PDF
documents = []

for file in glob.glob("DIC/*.pdf"):
    try:
        loader = PyPDFLoader(file)  # Retourne une liste de document (un pour chaque page)
        documents += loader.load()
    except Exception as e:
        print(f"Erreur survenue pour le fichier '{file}' : {e}")

In [4]:
# Première page du premier document PDF
documents[0]

Document(metadata={'producer': 'Actuate PDF Writer (Low Resolution) 2.1', 'creator': 'Actuate e.Reports', 'creationdate': '2022-07-29T08:11:45+01:00', 'title': '', 'subject': '', 'author': 'IDS GmbH - Analysis and Reporting Services', 'keywords': 'FR0010032326 (22.08.2022)', 'source': 'DIC/Allianz.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='\x00I\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00c\x00l\x00Ø\x00s\x00 \x00p\x00o\x00u\x00r\x00 \x00l \x19\x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\n\x00C\x00e\x00 \x00d\x00o\x00c\x00u\x00m\x00e\x00n\x00t\x00 \x00f\x00o\x00u\x00r\x00n\x00i\x00t\x00 \x00d\x00e\x00s\x00 \x00i\x00n\x00f\x00o\x00r\x00m\x00a\x00t\x00i\x00o\x00n\x00s\x00 \x00e\x00s\x00s\x00e\x00n\x00t\x00i\x00e\x00l\x00l\x00e\x00s\x00 \x00a\x00u\x00x\x00 \x00i\x00n\x00v\x00e\x00s\x00t\x00i\x00s\x00s\x00e\x00u\x00r\x00s\x00 \x00d\x00e\x00 \x00c\x00e\x00t\x00 \x00O\x00P\x00C\x00V\x00M\x00.\x00 \x00I\x00l\x00 \x00n\x00e\x00 \x00s

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Initialisation du séparateur de texte avec des paramètres spécifiques pour diviser le texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,  # Taille maximale des morceaux de texte
    chunk_overlap=60,  # Chevauchement entre les morceaux pour garder le contexte
    length_function=len,  # Fonction pour calculer la longueur des morceaux
    separators=["\n\n", "\n"]  # Séparateurs utilisés pour diviser le texte en morceaux
)

# Division du document en morceaux (chunks)
chunks = text_splitter.split_documents(documents=documents)

# Affichage du nombre de morceaux créés à partir du document PDF
print(f"{len(chunks)} chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.")

1721 chunks ont été créés par le splitter à partir des documents PDF du dossier DIC.


In [6]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [7]:
# Choisir un modèle d’embedding
#On normalise les embeddings afin d’améliorer la qualité et la stabilité du calcul de similarité (notamment la similarité cosinus) lors de la recherche vectorielle.
#modèle open-source multilingue car il comprend bien le français et offre de bonnes performances en recherche sémantique, tout en étant exécutable localement sans API externe.
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2", encode_kwargs={"normalize_embeddings" : True})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
#Créer la base Chroma (locale)
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db",  # stockage local
    collection_name="dic_collection_v2"  
)

In [9]:
#sauvegarde
vectorstore.persist()

/tmp/ipykernel_702/2251463224.py:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [21]:
#vérification
query = "Quels type d'assurance proposez vous?"
results = vectorstore.similarity_search(query)

for r in results:
    print(r.page_content)

Combien de temps dois-je le conserver et puis-je retirer de l’argent de façon anticipée ? 
DURÉE DE PLACEMENT RECOMMANDÉE : 5 ANS
Ce FCP pourrait ne pas convenir aux investisseurs qui prévoient de retirer leur apport dans les 5 ans. La durée de détention re commandée a pour objet de minimiser 
votre risque de perte en capital en cas de rachat après cette période même si celle-ci ne constitue pas une garantie. Vous pouvez par ailleurs procéder à tout moment 
au rachat de votre investissement, votre FCP ne prélevant aucune commission de rachat.
Comment puis-je formuler une réclamation ? 
Pour toute réclamation, vous pouvez contacter SG 29 HAUSSMANN, 29 boulevard Haussmann, 75009 Paris, ainsi que sur le site internet : 
https://sg29haussmann.societegenerale.fr.
Autres informations pertinentes  
Le FCP promeut des caractéristiques environnementales et/ou sociales au sens de l’article 8 du Règlement SFDR.  
Lorsque ce produit est utilisé comme support en unité de compte d’un contrat d’assur

In [22]:
## PARTIE 2 
import torch
from transformers import MistralCommonBackend, Ministral3ForCausalLM, FineGrainedFP8Config
from langchain_classic.chains import ConversationChain
from langchain_classic.chains.conversation.memory import ConversationBufferMemory

model_id = "mistralai/Ministral-3-14B-Instruct-2512"
tokenizer = MistralCommonBackend.from_pretrained(model_id)
model = Ministral3ForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=FineGrainedFP8Config(dequantize=True)
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights: 0it [00:00, ?it/s]

Ministral3ForCausalLM LOAD REPORT from: mistralai/Ministral-3-14B-Instruct-2512
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
language_model.model.layers.{0...39}.self_attn.q_proj.weight           | UNEXPECTED | 
language_model.model.layers.{0...39}.self_attn.k_proj.activation_scale | UNEXPECTED | 
vision_tower.transformer.layers.{0...23}.feed_forward.down_proj.weight | UNEXPECTED | 
language_model.model.layers.{0...39}.mlp.up_proj.activation_scale      | UNEXPECTED | 
language_model.model.layers.{0...39}.self_attn.v_proj.weight           | UNEXPECTED | 
language_model.model.layers.{0...39}.self_attn.k_proj.weight           | UNEXPECTED | 
vision_tower.transformer.layers.{0...23}.attention.k_proj.weight       | UNEXPECTED | 
language_model.model.layers.{0...39}.self_attn.o_proj.weight           | UNEXPECTED | 
language_model.model.layers.{0...39}.input_layerno

In [17]:
from langchain_huggingface import HuggingFacePipeline

pipeline = TextGenerationPipeline(
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=4096,
    do_sample=False,
    return_full_text=False,  # Très important ! On ne veut pas le prompt initial
)

# Création d'une instance de HuggingFacePipeline à partir de l'identifiant du modèle spécifié
llm = HuggingFacePipeline(pipeline=pipeline)

The model 'Mistral3ForConditionalGeneration' is not supported for . Supported models are ['AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausalLM', 'FlexOlmoForCausalLM', 'Fuy

In [26]:
from langchain_community.memory import ConversationBufferWindowMemory

memory = ConversationBufferMemory(return_messages=False)

prompt_template = """Tu es un assistant spécialisé en analyse de documents d’informations clés (DIC) pour un département ALM d’assurance vie.Réponds uniquement à partir du contexte fourni.
Si l’information est présente, cite la source (nom du document).
Si elle n’est pas présente dans les documents fournis, indique clairement que l’information n’est pas disponible dans les DIC..

Historique de la conversation :
{history}

Contexte :
{context}

Question :
{question}

Réponse :
"""

def rag_pipeline(query):
    # Récupérer l'historique
    history = memory.load_memory_variables({})["history"]
    
    # on recherche les documents
    retrieved_docs = vectorstore.similarity_search(query)
    
    # Ensuite, on injecte l'historique et les documents dans le prompt
    prompt = prompt_template.format(
        history=history,
        question=query,
        context="\n\n".join(doc.page_content for doc in retrieved_docs)
    )
    # Appel via llm (PAS pipeline direct)
    response = llm.invoke(prompt)
    
    # Sauvegarde dans la mémoire
    memory.save_context({"input": query}, {"output": response})
    
    return response

ImportError: cannot import name 'ConversationBufferWindowMemory' from 'langchain_community.memory' (/opt/conda/lib/python3.11/site-packages/langchain_community/memory/__init__.py)

In [16]:
query = """
Quel est l'objectif de gestion du FCP décrit dans le document ? ?
"""

# Effectuer une requête
response = rag_pipeline(query)
print(response)

Both `max_new_tokens` (=4096) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


L'objectif de gestion du **FCP "29 Haussmann Euro Crédit - Part C"** est de **surperformer l’indice ICE BofA Euro Corporate (ER00 Index)** sur une durée de placement recommandée **supérieure à 5 ans**, en s’exposant aux marchés internationaux de taux et de crédit. Cette gestion est mise en œuvre de manière discrétionnaire au sein de l’OPCVM.

*(Source : Document d’Informations Clés pour le FCP "29 Haussmann Euro Crédit - Part C")*


In [23]:
On utilise ConversationBufferWindowMemory(k=3) afin de conserver uniquement les derniers échanges pertinents pour le suivi conversationnel, tout en évitant d’alourdir inutilement le contexte et de dépasser la limite de tokens du modèle.
Dans un contexte financier et réglementaire comme celui des DIC, cette approche est préférable à une mémoire par résumé, car elle évite toute reformulation ou simplification automatique qui pourrait altérer la précision des informations.

SyntaxError: invalid character '’' (U+2019) (458845927.py, line 1)